In [1]:
from argilla_client import client, rg, workspace
from tool_extraction import SEED, artifacts


In [ ]:
settings = rg.Settings(
    guidelines="Extract information from the skills content or metadata.",
    fields=[
        rg.TextField(name="content", use_markdown=True),
    ],
    questions=[
        rg.LabelQuestion(
            name="included",
            labels={
                "yes": "Yes",
                "not_plain_text": "No. Body is not plain text (e.g. JSON, CSS)",
                "not_english": "No. Not English",
                "not_skill": "No. Not a skill",
                "not_tool": "No. No tool mentioned",
            },
            required=True,
            title="Is this skill included?",
        ),
        rg.SpanQuestion(
            name="mentions",
            field="content",
            labels=["tool", "publisher", "repository"],
            allow_overlapping=True,
            required=False,
            title="Mark each taught tool once, and its publisher and repository when the text names them",
            description=(
                "A tool is a product this skill teaches the agent to use, such as npm in a React skill. "
                "Do not mark the folder that hosts the skill, the repository that publishes the skill, "
                "a link that credits where the skill came from, the skill filename, or GitHub inside an address."
            ),
        ),
        rg.TextQuestion(
            name="manual_details",
            title="Publisher or repository the text does not name",
            description=(
                "Only for marked tools still missing a publisher or repository. "
                "One line per tool: `tool | publisher | repository`. "
                "Omit the tool name to bind by mark order. "
                "Leave a field empty when it is already marked. "
                "Repository is a full https project URL, or `none`. "
                "No `|` inside a value. "
                "Leave blank when every value is marked, or when included is not yes."
            ),
            required=False,
        ),
    ],
    metadata=[
        rg.TermsMetadataProperty(name="file_sha"),
        rg.TermsMetadataProperty(name="repo_full_name"),
        rg.TermsMetadataProperty(name="path"),
    ],
)

In [3]:
import polars as pl

content = pl.col("content").str.strip_chars().replace("", None)
is_json = content.str.json_path_match("$").is_not_null()

skills = (
    artifacts.filter(
        (pl.col("dedup_primary") == 1)
        & (pl.col("frontmatter_valid") == 1)
        & (pl.col("filename").str.to_lowercase() == "skill.md")
        & (pl.col("location_class").is_in(["canonical", "skills-dir"]))
    )
    .with_columns(content.alias("content"))
    .filter(pl.col("content").is_not_null() & ~is_json)
    .select("file_sha", "repo_full_name", "path", "content")
)

sample = skills.select(pl.all().sample(n=5000, seed=SEED)).collect()

In [4]:
dataset_name = "skills"
existing = client.datasets(dataset_name)
if existing is not None:
    existing.delete()

dataset = rg.Dataset(name=dataset_name, settings=settings, client=client, workspace=workspace)
dataset.create()
dataset.records.log(
    [
        {
            "id": row["file_sha"],
            "content": row["content"],
            "file_sha": row["file_sha"],
            "repo_full_name": row["repo_full_name"],
            "path": row["path"],
        }
        for row in sample.iter_rows(named=True)
    ]
)

Sending records...: 20batch [00:23,  1.17s/batch]                     


DatasetRecords(Dataset(id=UUID('de566121-2528-49f7-861d-ed955e5768e4') inserted_at=datetime.datetime(2026, 11, 10, 22, 15, 12, 871866) updated_at=datetime.datetime(2026, 11, 10, 22, 15, 12, 965924) name='skills' status='ready' guidelines='Extract information from the skills content or metadata.' allow_extra_metadata=False distribution=OverlapTaskDistributionModel(strategy='overlap', min_submitted=1) workspace_id=UUID('77dd63ed-30ef-4467-890a-2edf2ca80aa3') last_activity_at=datetime.datetime(2026, 11, 10, 22, 15, 12, 965924)))